# DeepWeeds — Phase B: Bước 3-5 (suy luận, chung kết, xlsx)

In [ ]:
import os, sys, platform
REPO_URL = "https://github.com/hdphuc127/K4-Track4-Day2-Deeplearning-Advance.git"
SUB = "submissions/2A202602796-HoDangPhuc/code"
ROOT = "/kaggle/working"
os.chdir(ROOT)
if not os.path.exists("lab"):
    !git clone -q {REPO_URL} lab
else:
    !git -C lab pull -q
CODE = f"{ROOT}/lab/{SUB}"
sys.path.insert(0, CODE)
!pip -q install timm openpyxl
import torch, timm
print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, hashlib
os.makedirs("data/labels", exist_ok=True)
# Ảnh: dataset Kaggle ashuraotsuki/deepweeds-fold0 (images.zip nguyên bản Zenodo, MD5 b7b30f96...). Có thể đã được giải nén sẵn.
zips = glob.glob("/kaggle/input/**/images.zip", recursive=True)
if zips:
    h = hashlib.md5()
    with open(zips[0], "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""): h.update(c)
    assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", h.hexdigest()
    os.makedirs("data/images", exist_ok=True)
    !unzip -q -n {zips[0]} -d data/images     # zip phẳng: 17.509 file .jpg ở gốc
    IMAGES_DIR = f"{ROOT}/data/images"
else:
    jp = glob.glob("/kaggle/input/**/*.jpg", recursive=True)
    assert jp, "không thấy ảnh trong /kaggle/input: gắn dataset ashuraotsuki/deepweeds-fold0"
    IMAGES_DIR = os.path.dirname(jp[0])
print("IMAGES_DIR", IMAGES_DIR, len(os.listdir(IMAGES_DIR)))
B_ = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"   # nhãn/fold nguyên bản của tác giả
for n in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{n}.csv {B_}/{n}.csv
!wc -l data/labels/*.csv

In [ ]:
LABELS_DIR = f"{ROOT}/data/labels"
OUT = f"{ROOT}/out"
os.makedirs(OUT, exist_ok=True)
import experiments as E, train as Tr, dataset as D
import pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
PATHS = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out_dir=f"{OUT}/runs",
             pred_dir=f"{OUT}/predictions", curves_dir=f"{OUT}/curves", num_workers=4)
EPOCHS = 12
def cfg(**kw): return Tr.Config(**{**PATHS, "epochs": EPOCHS, **kw})

In [ ]:
import glob, shutil
cands = sorted(set(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/runs", recursive=True)))
print("output các phase trước:", cands)
assert cands, "không thấy output phase A/C trong /kaggle/input"
for src_out in cands:
    for sub in ("runs", "predictions", "curves", "eda"):
        if os.path.isdir(f"{src_out}/{sub}"):
            shutil.copytree(f"{src_out}/{sub}", f"{OUT}/{sub}", dirs_exist_ok=True)
    for f in ("backbones.csv", "training.csv", "training_c.csv", "failed.txt"):
        if os.path.exists(f"{src_out}/{f}"): shutil.copy(f"{src_out}/{f}", f"{OUT}/{f}")
B = pd.read_csv(f"{OUT}/backbones.csv"); print(B.shape)

In [ ]:
import numpy as np, inference as I, benchmark as BM
from eval import compute_metrics
tr, va, te = D.load_split(LABELS_DIR)
BEST = "convnext_tiny"
COMBO = dict(mix='cutmix', loss='ls', label_smoothing=0.1, ema_decay=0.995)
FINAL_VIEWS = {"id": I.view_identity, "flip": I.view_hflip} if "flip" == "flip" else {"id": I.view_identity}
SPACE = "prob"
E.run_resume(cfg(exp_id="C13", backbone=BEST, seed=0, **COMBO))
Pil = E.collect_runs(PATHS["out_dir"], "T")   # pilot ResNet-50 (T00-T12)
Tdf = E.collect_runs(PATHS["out_dir"], "C")   # ablation chính ConvNeXt-T (C00-C13)
Tdf["delta_vs_C00"] = Tdf.val_macro_f1 - Tdf.loc[Tdf.exp_id == "C00", "val_macro_f1"].iloc[0]

In [ ]:
REF = cfg(exp_id="C13", backbone=BEST, seed=0, **COMBO)
dev = torch.device("cuda"); model = E.load_best(REF, dev)
va_loader = E._loader(REF, va)
V = {"I00 1-view": {"id": I.view_identity},
     "I01 flip": {"id": I.view_identity, "flip": I.view_hflip},
     "I02 5crop192": {"c": lambda x: I.views_multicrop(x, 192)},
     "I02b 5crop+flip192": {"c": lambda x: I.views_multicrop(x, 192, True)}}
rows = []
for name, views in V.items():
    for space in ("prob", "logit"):
        _, y, lg = E.view_logits(model, va_loader, dev, views)
        p = I.aggregate_views(list(lg.values()), space)
        m_ = compute_metrics(y, p.argmax(1), p)
        rows.append({"exp_id": name.split()[0], "method": name, "space": space, "K": len(views),
                     "macro_f1_val": m_["macro_f1"], "top1_val": m_["top1"], "ece_val": m_["ece"]})
for s_ in (224, 256, 288, 320):
    _, y, lg = E.view_logits(model, E._loader(REF, va, s_), dev, {"id": I.view_identity})
    p = I._softmax(lg["id"]); m_ = compute_metrics(y, p.argmax(1), p)
    rows.append({"exp_id": "I04", "method": f"res{s_}", "space": "prob", "K": 1, "macro_f1_val": m_["macro_f1"],
                 "top1_val": m_["top1"], "ece_val": m_["ece"]})
_, y, lg = E.view_logits(model, va_loader, dev, {"id": I.view_identity})
z = lg["id"]; Tt = I.fit_temperature(z, y)
for nm, p in (("trước", I._softmax(z)), ("sau", I.apply_temperature(z, Tt))):
    m_ = compute_metrics(y, p.argmax(1), p)
    rows.append({"exp_id": "I07", "method": f"temperature {nm} (T={Tt:.3f})", "space": "prob", "K": 1,
                 "macro_f1_val": m_["macro_f1"], "top1_val": m_["top1"], "ece_val": m_["ece"]})
Inf = pd.DataFrame(rows); Inf.to_csv(f"{OUT}/inference.csv", index=False); print(Inf)

In [ ]:
# Độ trễ đúng cách: warmup 10, synchronize, 100 lần; batch 1 và 32; fp32/amp/fp16; gộp BN
lat = []
for b in (1, 32):
    for dt in ("fp32", "amp", "fp16"):
        lat.append({**BM.latency_report(model, b, 224, dt), "fuse_bn": False})
fused = I.fuse_conv_bn(model)
for b in (1, 32):
    lat.append({**BM.latency_report(fused, b, 224, "fp32"), "fuse_bn": True})
lat.append({**BM.tta_latency(model, 2, batch_size=1, img_size=224, dtype="fp32"), "gpu": torch.cuda.get_device_name()})
Lat = pd.DataFrame(lat); Lat.to_csv(f"{OUT}/latency.csv", index=False); print(Lat)

In [ ]:
# Chung kết: huấn luyện KHÔNG ghi test; finalize() chạy test đúng một lần/seed; mốc T00 ghi test khi huấn luyện
for seed in (0, 1, 2):
    c = cfg(exp_id="F01", backbone=BEST, seed=seed, **COMBO)
    E.run_resume(c); E.finalize(c, FINAL_VIEWS, SPACE)
    E.run_resume(cfg(exp_id="C00", backbone=BEST, seed=seed, save_test_predictions=True))

In [ ]:
P = f"{OUT}/predictions"; EV = f"{CODE}/eval.py"; LB = LABELS_DIR
!python {EV} score --pred "{P}/F01_seed*_test.csv" --test-csv {LB}/test_subset0.csv --labels {LB}/labels.csv --tag F01 --out {OUT}/eval_out
!python {EV} score --pred "{P}/C00_seed*_test.csv" --test-csv {LB}/test_subset0.csv --labels {LB}/labels.csv --tag C00 --out {OUT}/eval_out
!python {EV} grade --final "{P}/F01_seed*_test.csv" --baseline "{P}/C00_seed*_test.csv" --uncal "{P}/F01uncal_seed*_test.csv" --final-val "{P}/F01_seed*_val.csv" --test-csv {LB}/test_subset0.csv --labels {LB}/labels.csv --latency-p95-ms {float(0) or 0} --out {OUT}/eval_out | tee {OUT}/grade.txt

In [ ]:
Final = E.collect_runs(PATHS["out_dir"], "F01")
Summ = pd.concat([B, Tdf, Pil]).sort_values("val_macro_f1", ascending=False).head(10)
E.write_xlsx({"Backbones": B, "Training": Tdf, "Training_pilot_resnet50": Pil, "Inference": Inf, "Final": Final, "Latency": Lat, "Summary": Summ},
             f"{OUT}/results.xlsx")
!find {OUT}/runs -name "best.pt" -delete
!du -sh {OUT}; ls {OUT}; ls {OUT}/eval_out